In [1]:
# %% [0] Config + pure helpers  (numpy / pandas / scipy / sklearn only -> unit-testable)
"""
Soil grain-size distribution from photos - pipelined version.

Design rules (each one fixes a concrete weakness of the first script):
  1. IDs never travel separately from values: every prediction is a DataFrame indexed by sample_id.
  2. Real validation: site hold-out and site x phone-swap CV, with a "predict the mean curve" baseline.
  3. Physical scale first: every photo is put on ONE pixel density, sharp test photos are degraded to
     the training condition (downscale + JPEG with the training photos' own quantisation tables).
  4. Two complementary branches, both Weibull-parametrised, both validated:
        A = DoG texture features -> Ridge -> Weibull      (few parameters, physical units)
        B = frozen EfficientNetV2-S embeddings (grayscale crops) -> EMD-trained linear head
  5. Stone measurement (SAM) is a separate, cached stage; its calibration is clipped and reported.
  6. Ablation submissions are written so the leaderboard can attribute any gain to a component.
"""
import re, io, json, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SEED = 0
np.random.seed(SEED)

# ---- switches -------------------------------------------------------------------------------
USE_CNN, USE_SAM = True, True
FORCE_RERUN_SAM = False          # False: reuse stone_cache.csv if it exists
STONE_DEF = "bbox"               # 'bbox' (what your best run used) | 'pca_major' | 'pca_minor' | 'equiv'
GATE_A63 = 0.02                  # redraw the curve only if >= 2% of the image is stones longer than 63 mm
K_CLIP = (0.8, 1.6)              # area->mass factor is clipped (Delesse prior is k = 1)
MIN_FINES = 1.0                  # floor (%) for the 20 mm point after a stone redraw
ALPHA_PRIOR, ALPHA_MARGIN, ALPHA_OVERRIDE = 0.5, 0.5, None   # weight of branch B in the blend
CNN_L2_GRID = (0.3, 3.0, 30.0)   # candidate strengths for the EMD-trained head (all validated)
CNN_PCA, K_CROPS, CROP_PX = 24, 12, 400
CNN_USE_COLOR = False            # grayscale = no phone colour shortcut
TARGET_PPM = TRAIN_LIKE_PPM = 4.6
SHARP_PPM = 6.0                  # photos sharper than this are "native resolution" test photos
SAM_PPM, SAM_BORDER, SAM_FULLRES_PPM = 2.0, 0.12, 12.0
CAM_EDGE, CAM_A52 = "Motorola Edge", "Samsung A52"

# ---- competition grid / metric -----------------------------------------------------------------
SIZES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200])
SIZE_COLS = ["0.002", "0.0063", "0.02", "0.063", "0.2", "0.63", "2", "6.3", "20", "63", "200"]
W_EMD = np.diff(np.log10(SIZES))                 # left-endpoint rule, as in your script's comment
LOGD10 = np.log(SIZES[:10])
I20, I63 = SIZE_COLS.index("20"), SIZE_COLS.index("63")


def emd(y_true, y_pred):
    a = np.asarray(y_true, float)[..., :10]
    b = np.asarray(y_pred, float)[..., :10]
    return (np.abs(a - b) * W_EMD).sum(-1)


def postprocess(curve):
    c = np.maximum.accumulate(np.clip(np.asarray(curve, float), 0, 100), axis=-1)
    c[..., -1] = 100.0
    return c


def weibull_curve(lnb, c, d=SIZES):
    lnb = np.asarray(lnb, float)[..., None]
    c = np.maximum(np.asarray(c, float), 0.02)[..., None]
    return 100.0 * (1.0 - np.exp(-np.exp(c * (np.log(d) - lnb))))


def fit_weibull_label(y10):
    """(ln b, c) that minimise the competition EMD for one measured curve (grid start + Nelder-Mead)."""
    y10 = np.asarray(y10, float)[:10]
    g_lnb, g_c = np.linspace(-6, 4, 41), np.linspace(0.1, 2.0, 39)
    L, C = np.meshgrid(g_lnb, g_c, indexing="ij")
    F = 100 * (1 - np.exp(-np.exp(C[..., None] * (LOGD10 - L[..., None]))))
    i, j = np.unravel_index((np.abs(F - y10) * W_EMD).sum(-1).argmin(), L.shape)
    f = lambda p: (np.abs(100 * (1 - np.exp(-np.exp(max(p[1], 0.02) * (LOGD10 - p[0])))) - y10) * W_EMD).sum()
    return minimize(f, [L[i, j], C[i, j]], method="Nelder-Mead").x


def make_sites(ids, gap=12):
    """Group soils from one location: same letter prefix, numbers within `gap` (heuristic, see report)."""
    parsed = []
    for s in ids:
        m = re.match(r"^([A-Za-z]+)(\d+)$", s)
        parsed.append((m.group(1), int(m.group(2)), s) if m else (s, -10**6, s))
    parsed.sort()
    site, last, cur = {}, None, 0
    for letter, num, s in parsed:
        if last is None or letter != last[0] or num - last[1] > gap:
            cur += 1
        site[s] = f"site{cur:02d}"
        last = (letter, num)
    return pd.Series(site)


def soil_weights(sids):
    """Every soil gets the same total weight, split evenly over its rows. Sums to 1."""
    s = pd.Series(np.asarray(sids))
    n = s.map(s.value_counts()).to_numpy(float)
    w = 1.0 / n
    return w / w.sum()


def aggregate_soil(curves, sids):
    """Rows -> one curve per soil. Returns a DataFrame INDEXED BY sample_id (never an id list)."""
    df = pd.DataFrame(np.asarray(curves, float), columns=SIZE_COLS)
    df["sid"] = np.asarray(sids)
    out = df.groupby("sid", sort=True).mean()
    return pd.DataFrame(postprocess(out.to_numpy()), index=out.index, columns=SIZE_COLS)


def d50_of(curve):
    return float(np.exp(np.interp(50.0, np.asarray(curve, float), np.log(SIZES))))


# ---- models (uniform interface: fit(X, Y11, wb, w) / predict(X) -> (n, 11)) -----------------------
class WeibullRidge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, Y, wb, w):
        self.sc = StandardScaler().fit(X)
        self.m = Ridge(alpha=self.alpha).fit(self.sc.transform(X), wb, sample_weight=w * len(w))
        return self

    def predict(self, X):
        p = self.m.predict(self.sc.transform(X))
        return weibull_curve(p[:, 0], p[:, 1])


class MeanCurve:
    def fit(self, X, Y, wb, w):
        self.c = (Y * w[:, None]).sum(0) / w.sum()
        return self

    def predict(self, X):
        return np.tile(self.c, (len(X), 1))


class EmdHead:
    """Standardise -> PCA -> linear map to (ln b, c) -> Weibull; trained on the exact metric (numpy Adam)."""

    def __init__(self, n_pca=24, l2=3.0, epochs=300, lr=0.03, seed=0):
        self.n_pca, self.l2, self.epochs, self.lr, self.seed = n_pca, l2, epochs, lr, seed

    @staticmethod
    def forward(P, Wt, b):
        z = P @ Wt + b
        lnb, c = z[:, 0], np.logaddexp(0, z[:, 1]) + 0.02
        u = np.exp(np.clip(c[:, None] * (LOGD10[None, :] - lnb[:, None]), -40, 40))
        return z, lnb, c, u, 100.0 * (1 - np.exp(-u))

    def loss_grad(self, P, Y10, w, Wt, b):
        z, lnb, c, u, F = self.forward(P, Wt, b)
        r = F - Y10
        loss = (w * (np.abs(r) * W_EMD).sum(1)).sum() + self.l2 * (Wt ** 2).sum()
        s = np.sign(r) * W_EMD[None, :]
        eu = np.exp(-u)
        g0 = (s * (-100.0 * c[:, None] * u * eu)).sum(1)
        g1 = (s * (100.0 * u * eu * (LOGD10[None, :] - lnb[:, None]))).sum(1) / (1 + np.exp(-z[:, 1]))
        G = np.stack([g0, g1], 1) * w[:, None]
        return loss, P.T @ G + 2 * self.l2 * Wt, G.sum(0)

    def _prep(self, X):
        P = self.pca.transform(self.sc.transform(X))
        return P / self.pscale

    def fit(self, X, Y, wb, w):
        self.sc = StandardScaler().fit(X)
        Z = self.sc.transform(X)
        k = int(max(1, min(self.n_pca, Z.shape[1], Z.shape[0] - 1)))
        self.pca = PCA(k, random_state=self.seed).fit(Z)
        P0 = self.pca.transform(Z)
        self.pscale = float(P0[:, 0].std() + 1e-9)
        P, w = P0 / self.pscale, w / w.sum()
        m_lnb = float((wb[:, 0] * w).sum())
        m_c = max(float((wb[:, 1] * w).sum()) - 0.02, 0.05)
        Wt, b = np.zeros((k, 2)), np.array([m_lnb, np.log(np.expm1(m_c))])
        mW, vW, mb, vb = np.zeros_like(Wt), np.zeros_like(Wt), np.zeros(2), np.zeros(2)
        b1, b2, eps = 0.9, 0.999, 1e-8
        for t in range(1, self.epochs + 1):
            _, gW, gb = self.loss_grad(P, Y[:, :10], w, Wt, b)
            mW, vW = b1 * mW + (1 - b1) * gW, b2 * vW + (1 - b2) * gW ** 2
            mb, vb = b1 * mb + (1 - b1) * gb, b2 * vb + (1 - b2) * gb ** 2
            Wt -= self.lr * (mW / (1 - b1 ** t)) / (np.sqrt(vW / (1 - b2 ** t)) + eps)
            b -= self.lr * (mb / (1 - b1 ** t)) / (np.sqrt(vb / (1 - b2 ** t)) + eps)
        self.Wt, self.b = Wt, b
        return self

    def predict(self, X):
        _, _, _, _, F = self.forward(self._prep(X), self.Wt, self.b)
        out = np.zeros((len(X), 11))
        out[:, :10] = F
        out[:, 10] = 100.0
        return out


class FeatureSet:
    """X has one row per (photo, crop). row_photo maps each row to an index in `photos`."""

    def __init__(self, name, X, row_photo):
        self.name, self.X, self.row_photo = name, np.asarray(X, float), np.asarray(row_photo, int)


def fit_predict_rows(fs, make_model, tr_photos, te_photos):
    m_tr, m_te = np.isin(fs.row_photo, tr_photos), np.isin(fs.row_photo, te_photos)
    sid_tr = photos.sample_id.to_numpy()[fs.row_photo[m_tr]]
    Y = labels.loc[sid_tr, SIZE_COLS].to_numpy(float)
    wb = wb_labels.loc[sid_tr].to_numpy(float)
    model = make_model().fit(fs.X[m_tr], Y, wb, soil_weights(sid_tr))
    return model.predict(fs.X[m_te]), photos.sample_id.to_numpy()[fs.row_photo[m_te]], fs.row_photo[m_te]


def cv_items(fs, make_model, scheme):
    """Out-of-fold soil curves. scheme 'site': hold out a whole site. 'xphone': also train on one phone,
    predict the other (both directions) - a rehearsal of the train->test camera shift."""
    sid, cam, split = photos.sample_id.to_numpy(), photos.camera.to_numpy(), photos.split.to_numpy()
    tr_all = np.where(split == "train")[0]
    items = {}
    for site in sorted(SITE.unique()):
        held = np.isin(sid, list(SITE.index[SITE == site]))
        if scheme == "site":
            jobs = [("all", tr_all[~held[tr_all]], tr_all[held[tr_all]])]
        else:
            jobs = []
            for cx, cy in [(CAM_EDGE, CAM_A52), (CAM_A52, CAM_EDGE)]:
                tr = tr_all[(~held[tr_all]) & (cam[tr_all] == cx)]
                te = tr_all[held[tr_all] & (cam[tr_all] == cy)]
                if len(tr) and len(te):
                    jobs.append((f"{cx[:3]}>{cy[:3]}", tr, te))
        for tag, tr, te in jobs:
            curves, sids, _ = fit_predict_rows(fs, make_model, tr, te)
            soil = aggregate_soil(curves, sids)
            for s_, row in soil.iterrows():
                items[f"{s_}|{tag}"] = row.to_numpy()
    df = pd.DataFrame(items, index=SIZE_COLS).T
    meta = pd.Series(df.index.str.split("|").str[0], index=df.index)
    return df, meta


def per_soil_emd(curves_df, meta):
    e = emd(labels.loc[meta.to_numpy(), SIZE_COLS].to_numpy(float), postprocess(curves_df.to_numpy()))
    return pd.Series(e, index=curves_df.index).groupby(meta.to_numpy()).mean()


def stone_correct(curve, a63, a20, k):
    """Redraw the 20 / 63 mm points from measured stone area; keep the predicted shape of the finer matrix."""
    c = np.asarray(curve, float).copy()
    new20 = min(max(100.0 - k * 100.0 * a20, MIN_FINES), 100.0)      # never claim a soil has ~0% fines
    new63 = min(max(100.0 - k * 100.0 * a63, new20), 100.0)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1:
        c[: I20 + 1] *= scale          # = finer-than-20 matrix keeps its internal gradation
    c[I20], c[I63], c[-1] = new20, new63, 100.0
    return postprocess(c)


def check_submission(df, sample_ids):
    assert list(df.index) == list(sample_ids), "row order / ids differ from sample_submission"
    v = df[SIZE_COLS].to_numpy()
    assert not np.isnan(v).any()
    assert ((v >= 0) & (v <= 100)).all() and (np.diff(v, axis=1) >= -1e-9).all() and (v[:, -1] == 100).all()


# %% [1] Index photos - physical scale from each file's OWN size
import cv2
from PIL import Image, ImageOps

INPUT_DIR = sorted(Path("/kaggle/input").glob("**/Training_labels_updated.csv"))[0].parent
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

CAMERA_RE = re.compile(r"^(Motorola_Edge_60_fusion|Motorola_Edge|Samsung_A52|iPhone_?14|iPhone_?16)_(.+?)(?:_\d+| \(\d+\))?$")
CAMERA_KEY = {"Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion", "Motorola_Edge": CAM_EDGE,
              "Samsung_A52": CAM_A52, "iPhone14": "iPhone 14", "iPhone_14": "iPhone 14",
              "iPhone16": "iPhone 16", "iPhone_16": "iPhone 16"}


def normalize_id(raw):
    s = unicodedata.normalize("NFC", raw).replace("ü", "ue").replace("ö", "oe").replace("ä", "ae")
    return s.replace(",", "_")


labels = pd.read_csv(INPUT_DIR / "Training_labels_updated.csv", dtype={"sample_id": str}).set_index("sample_id")
sample_sub = pd.read_csv(INPUT_DIR / "sample_submission.csv", dtype={"sample_id": str})
ppm_tab = pd.read_csv(INPUT_DIR / "ppm_updated.csv").set_index("phone")

rows = []
for split, pattern in [("train", "Training-All_Photos*"), ("test", "Test_All_Photos*")]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None:
            raise ValueError(f"unexpected file name: {p.name}")
        with Image.open(p) as im:
            w, h = im.size
        rows.append({"split": split, "path": p, "camera": CAMERA_KEY[m.group(1)],
                     "sample_id": normalize_id(m.group(2)), "w": w, "h": h})
photos = pd.DataFrame(rows)
photos["ppm_eff"] = [ppm_tab.loc[c, "ppm"] * max(w, h) / ppm_tab.loc[c, "width"]
                     for c, w, h in zip(photos.camera, photos.w, photos.h)]
assert sorted(set(photos.query("split=='train'").sample_id)) == sorted(labels.index)
assert set(photos.query("split=='test'").sample_id) == set(sample_sub.sample_id)
print(photos.groupby(["split", "camera"]).agg(photos=("path", "size"), px_per_mm=("ppm_eff", "median")).round(1))
_tr_ppm = photos.query("split=='train'").ppm_eff
if not _tr_ppm.between(3.5, 5.5).all():
    warnings.warn("Some TRAIN photos are not near 4.6 px/mm: re-check ppm_updated.csv vs the file sizes!")

# %% [2] Targets: Weibull fit of every label with the exact metric, site groups, soil types
wb_labels = labels[SIZE_COLS[:10]].apply(lambda r: pd.Series(fit_weibull_label(r.to_numpy()), index=["lnb", "c"]), axis=1)
wb_labels["c"] = wb_labels["c"].clip(lower=0.02)
_fit = weibull_curve(wb_labels.lnb.to_numpy(), wb_labels.c.to_numpy())
print("Weibull representation floor (EMD of fitted labels):", emd(labels[SIZE_COLS].to_numpy(), _fit).mean().round(2))
SITE = make_sites(list(labels.index))
print("sites:", SITE.nunique(), "| largest:", SITE.value_counts().head(3).to_dict())
_f2, _f063 = labels["2"], labels["0.063"]
SOIL_TYPE = pd.Series(np.where(_f2 < 50, "gravel", np.where(_f063 >= 50, "fine", "sand")), index=labels.index)
print(SOIL_TYPE.value_counts().to_dict())

# %% [3] Image cache on ONE physical scale + features
qtables = None
try:
    with Image.open(photos.query("split=='train'").path.iloc[0]) as _im0:
        qtables = getattr(_im0, "quantization", None)
except Exception:
    pass


def degrade_to_train_like(a):
    buf = io.BytesIO()
    try:
        Image.fromarray(a).save(buf, format="JPEG", qtables=qtables) if qtables else Image.fromarray(a).save(buf, format="JPEG", quality=75)
    except Exception:
        buf = io.BytesIO(); Image.fromarray(a).save(buf, format="JPEG", quality=75)
    return np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert("RGB"))


def load_scaled(path, ppm_eff):
    """RGB uint8 at TARGET_PPM, landscape. Sharp (test) photos are first degraded to the training condition."""
    a = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))
    if ppm_eff > SHARP_PPM:
        f = TRAIN_LIKE_PPM / ppm_eff
        a = cv2.resize(a, (round(a.shape[1] * f), round(a.shape[0] * f)), interpolation=cv2.INTER_AREA)
        a = degrade_to_train_like(a)
        ppm_eff = TRAIN_LIKE_PPM
    f = TARGET_PPM / ppm_eff
    if abs(f - 1) > 0.02:                                     # no needless resampling of already-4.6 photos
        a = cv2.resize(a, (round(a.shape[1] * f), round(a.shape[0] * f)), interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]:
        a = np.ascontiguousarray(np.rot90(a))
    return a


RGB_CACHE = [load_scaled(r.path, r.ppm_eff) for r in photos.itertuples()]
GRAY_CACHE = [cv2.cvtColor(a, cv2.COLOR_RGB2GRAY) for a in RGB_CACHE]
if not CNN_USE_COLOR:
    RGB_CACHE = None
print("cached", len(GRAY_CACHE), "photos at", TARGET_PPM, "px/mm")

DOG_SIGMAS_MM = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]


def dog_bands(gray):
    H, W = gray.shape
    c = int(0.25 * min(H, W))
    g = gray[c:H - c, c:W - c].astype(np.float32)
    blurs = [g] + [cv2.GaussianBlur(g, (0, 0), s * TARGET_PPM) for s in DOG_SIGMAS_MM]
    return g, [blurs[i] - blurs[i + 1] for i in range(len(DOG_SIGMAS_MM))]


def dog_raw6(gray):                      # = the first script's feature (mean |band|, 6 numbers)
    _, bands = dog_bands(gray)
    return np.array([np.abs(b).mean() for b in bands])


def dog_ratio5(gray):                    # contrast-share per band, log; skips the noise-dominated first band
    g, bands = dog_bands(gray)
    return np.log(np.array([b.std() for b in bands[1:]]) / (g.std() + 1e-6) + 1e-6)


_idx = np.arange(len(photos))
FEATURES = {
    "A_raw6": FeatureSet("A_raw6", np.stack([dog_raw6(g) for g in GRAY_CACHE]), _idx),
    "A_ratio5": FeatureSet("A_ratio5", np.stack([dog_ratio5(g) for g in GRAY_CACHE]), _idx),
}


def build_cnn_features():
    import tensorflow as tf
    tf.random.set_seed(SEED)
    net = tf.keras.applications.EfficientNetV2S(include_top=False, weights="imagenet", pooling="avg",
                                                input_shape=(CROP_PX, CROP_PX, 3))
    is_train = (photos.split.to_numpy() == "train")
    X, rp, batch, bp = [], [], [], []

    def flush():
        if batch:
            X.append(net.predict(np.stack(batch).astype("float32"), verbose=0)); rp.extend(bp)
            batch.clear(); bp.clear()

    for i in range(len(photos)):
        rng = np.random.default_rng(SEED * 100003 + i)
        src = RGB_CACHE[i] if CNN_USE_COLOR else GRAY_CACHE[i]
        H, W = src.shape[:2]
        for _ in range(K_CROPS):
            x0 = int(rng.integers(int(0.15 * W), max(int(0.85 * W) - CROP_PX, int(0.15 * W)) + 1))
            y0 = int(rng.integers(int(0.15 * H), max(int(0.85 * H) - CROP_PX, int(0.15 * H)) + 1))
            x0, y0 = min(x0, W - CROP_PX), min(y0, H - CROP_PX)
            a = src[y0:y0 + CROP_PX, x0:x0 + CROP_PX].astype("float32")
            a = np.rot90(a, int(rng.integers(0, 4)))
            if rng.random() < 0.5:
                a = a[:, ::-1]
            if is_train[i]:                                 # jitter on the 0-255 scale (the model rescales inside)
                a = np.clip((a - a.mean()) * rng.uniform(0.85, 1.15) + a.mean() + rng.uniform(-20, 20), 0, 255)
            if a.ndim == 2:
                a = np.repeat(a[..., None], 3, axis=2)
            batch.append(np.ascontiguousarray(a)); bp.append(i)
            if len(batch) == 64:
                flush()
    flush()
    tf.keras.backend.clear_session()
    return FeatureSet("cnn", np.concatenate(X), np.array(rp))


if USE_CNN:
    try:
        _cnn = build_cnn_features()
        for l2 in CNN_L2_GRID:
            FEATURES[f"B_l2_{l2:g}"] = _cnn
        print("CNN embeddings:", _cnn.X.shape)
    except Exception as e:               # no TF / no weights offline: continue with branch A only
        warnings.warn(f"CNN branch disabled: {e}")
        USE_CNN = False

MODELS = {"MEAN": ("A_raw6", lambda: MeanCurve()),
          "A_raw6": ("A_raw6", lambda: WeibullRidge(1.0)),
          "A_ratio5": ("A_ratio5", lambda: WeibullRidge(1.0))}
if USE_CNN:
    for l2 in CNN_L2_GRID:
        MODELS[f"B_l2_{l2:g}"] = (f"B_l2_{l2:g}", (lambda l2=l2: EmdHead(CNN_PCA, l2, seed=SEED)))

# %% [4] Honest validation: site hold-out and site x phone-swap, versus the mean-curve baseline
CV = {}
for name, (fkey, factory) in MODELS.items():
    CV[name] = {sch: cv_items(FEATURES[fkey], factory, sch) for sch in ("site", "xphone")}

table = pd.DataFrame({n: {sch: per_soil_emd(*CV[n][sch]).mean() for sch in ("site", "xphone")} for n in CV}).T
table["combined"] = 0.5 * table["site"] + 0.5 * table["xphone"]
print("\nCross-validated EMD (lower is better; MEAN = always the average training curve)")
print(table.round(2).sort_values("combined"))


def best_of(prefix):
    c = table[[i.startswith(prefix) for i in table.index]]
    return c["combined"].idxmin() if len(c) else None


BEST_A, BEST_B = best_of("A_"), (best_of("B_") if USE_CNN else None)
print("chosen A:", BEST_A, "| chosen B:", BEST_B)

ALPHA = 0.0
if BEST_B:
    alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
    sc = {}
    for a in alphas:
        s = []
        for sch in ("site", "xphone"):
            (dfa, meta), (dfb, _) = CV[BEST_A][sch], CV[BEST_B][sch]
            assert list(dfa.index) == list(dfb.index)
            s.append(per_soil_emd(pd.DataFrame(postprocess(a * dfb.to_numpy() + (1 - a) * dfa.to_numpy()), index=dfa.index,
                                               columns=SIZE_COLS), meta).mean())
        sc[a] = 0.5 * sum(s)
    print("blend CV by weight on B:", {k: round(v, 2) for k, v in sc.items()})
    best_a = min(sc, key=sc.get)
    ALPHA = ALPHA_OVERRIDE if ALPHA_OVERRIDE is not None else (ALPHA_PRIOR if sc[ALPHA_PRIOR] <= sc[best_a] + ALPHA_MARGIN else best_a)
print("blend weight on B:", ALPHA)

_df, _meta = CV[BEST_A]["site"]
_s = per_soil_emd(_df, _meta)
print("\nBest A, site hold-out, by soil type:", _s.groupby(SOIL_TYPE.loc[_s.index]).mean().round(1).to_dict())
print("CAUTION: this CV only rehearses Motorola<->Samsung. The iPhone/field shift cannot be validated with these labels.")

# %% [5] Final fit on all training photos, predict test photos (soil-indexed DataFrames)
tr_idx = np.where(photos.split.to_numpy() == "train")[0]
te_idx = np.where(photos.split.to_numpy() == "test")[0]


def predict_test(name):
    fkey, factory = MODELS[name]
    curves, sids, rowph = fit_predict_rows(FEATURES[fkey], factory, tr_idx, te_idx)
    soil = aggregate_soil(curves, sids)
    ph = pd.DataFrame(curves, columns=SIZE_COLS); ph["photo"] = rowph; ph["sid"] = sids
    ph = ph.groupby("photo").agg({**{c: "mean" for c in SIZE_COLS}, "sid": "first"})
    spread = {s: float(np.mean([emd(r[SIZE_COLS].to_numpy(float), soil.loc[s].to_numpy()) for _, r in g.iterrows()]))
              for s, g in ph.groupby("sid")}
    return soil, pd.Series(spread)


predA, spreadA = predict_test(BEST_A)
predB = predict_test(BEST_B)[0] if BEST_B else None
if predB is not None:
    assert list(predA.index) == list(predB.index)
pred_blend = predA if predB is None else pd.DataFrame(
    postprocess(ALPHA * predB.to_numpy() + (1 - ALPHA) * predA.to_numpy()), index=predA.index, columns=SIZE_COLS)

# %% [6] Stone measurement with SAM (cached) + calibration
CACHE = OUTPUT_DIR / "stone_cache.csv"


def stone_geometry(m):
    ys, xs = np.nonzero(m)
    bbox = max(ys.max() - ys.min(), xs.max() - xs.min()) / SAM_PPM
    ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([xs, ys]))))[::-1]
    return {"bbox": bbox, "pca_major": 4 * np.sqrt(max(ev[0], 0)) / SAM_PPM,
            "pca_minor": 4 * np.sqrt(max(ev[1], 0)) / SAM_PPM, "equiv": 2 * np.sqrt(m.sum() / np.pi) / SAM_PPM}


def stones_from_masks(masks, tot):
    order = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum()))
    taken, out = np.zeros_like(masks[0], bool), []
    for i in order:
        m = masks[i]; sz = int(m.sum())
        if sz < (8.0 * SAM_PPM) ** 2 or sz > 0.5 * tot or (m & taken).sum() > 0.5 * sz:
            continue
        g = stone_geometry(m)
        if sz / (np.pi * (g["bbox"] * SAM_PPM / 2) ** 2 + 1e-6) < 0.35:
            continue
        taken |= m
        out.append({"area_frac": sz / tot, **g})
    return out


def load_work(path, ppm):
    a = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))
    a = cv2.resize(a, None, fx=SAM_PPM / ppm, fy=SAM_PPM / ppm, interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]:
        a = np.rot90(a)
    H, W = a.shape[:2]; c = SAM_BORDER
    return np.ascontiguousarray(a[int(H * c):int(H * (1 - c)), int(W * c):int(W * (1 - c))])


def measure_stones():
    import torch
    from transformers import pipeline
    gen = pipeline("mask-generation", model="facebook/sam-vit-base", device=0 if torch.cuda.is_available() else -1)
    rec = []
    for r in photos[photos.ppm_eff >= SAM_FULLRES_PPM].itertuples():
        rgb = load_work(r.path, r.ppm_eff)
        out = gen(Image.fromarray(rgb), points_per_batch=64, pred_iou_thresh=0.86, stability_score_thresh=0.9)
        st = stones_from_masks([np.asarray(m).astype(bool) for m in out["masks"]], rgb.shape[0] * rgb.shape[1])
        row = {"sample_id": r.sample_id, "split": r.split, "n": len(st), "cover": sum(s["area_frac"] for s in st)}
        for d in ("bbox", "pca_major", "pca_minor", "equiv"):
            row[f"A63_{d}"] = sum(s["area_frac"] for s in st if s[d] > 63)
            row[f"A20_{d}"] = sum(s["area_frac"] for s in st if s[d] > 20)
        rec.append(row)
    return pd.DataFrame(rec)


stone_ms, K_CAL = None, 1.0
if USE_SAM:
    try:
        meas = pd.read_csv(CACHE, dtype={"sample_id": str}) if (CACHE.exists() and not FORCE_RERUN_SAM) else measure_stones()
        meas.to_csv(CACHE, index=False)
        stone_ms = meas.groupby("sample_id").mean(numeric_only=True)
        a20_cal = stone_ms.loc["H374", f"A20_{STONE_DEF}"] if "H374" in stone_ms.index else 0.0
        if a20_cal > 0.02:
            K_CAL = float(np.clip((100.0 - labels.loc["H374", "20"]) / (100.0 * a20_cal), *K_CLIP))
        else:
            warnings.warn("H374 calibration unusable (A20 ~ 0): using Delesse k = 1")
        print(f"stone definition={STONE_DEF} | k={K_CAL:.3f} (clipped to {K_CLIP}) | calibration soil H374 A20={a20_cal:.3f}")
        print(stone_ms.loc[[s for s in stone_ms.index if s in set(sample_sub.sample_id)],
                           ["n", "cover", f"A63_{STONE_DEF}", f"A20_{STONE_DEF}"]].round(3))
    except Exception as e:
        warnings.warn(f"SAM stage disabled: {e}")
        USE_SAM = False


def with_stones(curves_df):
    if not USE_SAM or stone_ms is None:
        return curves_df
    out = {}
    for sid in curves_df.index:
        a63, a20 = (float(stone_ms.loc[sid, f"A63_{STONE_DEF}"]), float(stone_ms.loc[sid, f"A20_{STONE_DEF}"])) \
            if sid in stone_ms.index else (0.0, 0.0)
        out[sid] = stone_correct(curves_df.loc[sid].to_numpy(), a63, a20, K_CAL) if a63 >= GATE_A63 else curves_df.loc[sid].to_numpy()
    return pd.DataFrame(out, index=SIZE_COLS).T


# %% [7] Submissions: primary + ablations (so the leaderboard can attribute the gain)
ids = list(sample_sub.sample_id)
variants = {"sub_1_A_only": predA, "sub_2_A_stones": with_stones(predA)}
if predB is not None:
    variants["sub_3_blend_nostones"] = pred_blend
    variants["sub_4_blend_stones"] = with_stones(pred_blend)
    variants["sub_5_B_stones"] = with_stones(predB)
primary = variants.get("sub_4_blend_stones", variants["sub_2_A_stones"])

for name, df in variants.items():
    df = df.loc[ids, SIZE_COLS]
    df = pd.DataFrame(postprocess(df.to_numpy()), index=ids, columns=SIZE_COLS)
    check_submission(df, ids)
    out = df.reset_index().rename(columns={"index": "sample_id"})
    out.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, float_format="%.4f")
prim = pd.DataFrame(postprocess(primary.loc[ids, SIZE_COLS].to_numpy()), index=ids, columns=SIZE_COLS)
check_submission(prim, ids)
prim.reset_index().rename(columns={"index": "sample_id"}).to_csv(OUTPUT_DIR / "submission.csv", index=False, float_format="%.4f")

diag = pd.DataFrame({"n_photos": photos[photos.split == "test"].groupby("sample_id").size(),
                     "photo_spread(A)": spreadA.round(1),
                     "d50_mm": pd.Series({s: round(d50_of(prim.loc[s]), 3) for s in prim.index}),
                     "<0.063mm_%": prim["0.063"].round(1), "<2mm_%": prim["2"].round(1), "<63mm_%": prim["63"].round(1)})
print(diag.loc[ids].to_string())
json.dump({"chosen_A": BEST_A, "chosen_B": BEST_B, "alpha_B": ALPHA, "k": K_CAL, "stone_def": STONE_DEF,
           "cv": table.round(3).to_dict()}, open(OUTPUT_DIR / "run_summary.json", "w"), indent=2)
print("\nwritten:", sorted(p.name for p in OUTPUT_DIR.glob("sub*.csv")))

                               photos  px_per_mm
split camera                                    
test  iPhone 14                    14       13.9
      iPhone 16                    21       19.5
train Motorola Edge                69        4.6
      Motorola Edge 60 Fusion       3       12.5
      Samsung A52                  55        4.6
Weibull representation floor (EMD of fitted labels): 7.65
sites: 13 | largest: {'site06': 6, 'site11': 3, 'site03': 3}
{'gravel': 11, 'fine': 7, 'sand': 6}


/tmp/ipykernel_24/3874772633.py:308: UserWarning: Some TRAIN photos are not near 4.6 px/mm: re-check ppm_updated.csv vs the file sizes!
  warnings.warn("Some TRAIN photos are not near 4.6 px/mm: re-check ppm_updated.csv vs the file sizes!")


cached 162 photos at 4.6 px/mm


I0000 00:00:1793330126.137985      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793330126.141748      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


2026-10-30 03:15:51.121472: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 03:15:51.295118: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 03:15:52.477185: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 03:15:52.662952: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-30 03:15:54.384352: E external/local_xla/xla/stream_

... (4 outputs left out)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


stone definition=bbox | k=1.305 (clipped to (0.8, 1.6)) | calibration soil H374 A20=0.180
                                n  cover  A63_bbox  A20_bbox
sample_id                                                   
HPC_Airbus BS10-4bis7       7.333  0.020     0.000     0.004
HPC_Airbus BS12-5bis8       2.667  0.008     0.000     0.000
HPC_Airbus BS6-3            3.000  0.009     0.000     0.000
HPC_Audorfring             36.800  0.157     0.000     0.048
HPC_Kleinkummerfeld 18-3    0.333  0.001     0.000     0.000
HPC_Kleinkummerfeld 2-2     2.333  0.006     0.000     0.000
HPC_Kleinkummerfeld 2-3     1.333  0.005     0.000     0.000
HPC_Kleinkummerfeld 9-4     1.000  0.003     0.000     0.002
HPC_Muenster_BS6_9_0-10m   64.400  0.838     0.267     0.724
HPC_Testfeld Lidl WHV     104.250  0.635     0.000     0.407
                          n_photos  photo_spread(A)  d50_mm  <0.063mm_%  <2mm_%  <63mm_%
HPC_Airbus BS6-3                 3              7.4   0.168        27.1    95.3    100.0
